In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [21]:

# Load dataset
df = pd.read_csv("Dataset.csv")

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [6]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [7]:
df.tail()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y
613,LP002990,Female,No,0,Graduate,Yes,4583,0.0,133.0,360.0,0.0,Semiurban,N


In [11]:
# Handle missing values for numerical columns
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mean(), inplace=True)
df['Credit_History'].fillna(df['Credit_History'].mode()[0], inplace=True)
import warnings
# Ignore warnings
warnings.filterwarnings("ignore")

In [12]:
# Handle missing values for categorical columns
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    df[col].fillna(df[col].mode()[0], inplace=True)

df['Dependents'].replace({'3+': 3}, inplace=True)
df['Dependents'] = df['Dependents'].astype(int)

In [13]:
# Feature Engineering
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Total_Income_log'] = np.log(df['Total_Income'] + 1)
df['LoanAmount_log'] = np.log(df['LoanAmount'] + 1)

In [14]:
encoder = LabelEncoder()
categorical_columns = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

In [15]:
# Feature selection
X = df.drop(columns=['Loan_ID', 'Loan_Status'])
y = df['Loan_Status']

In [16]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
# Train models
models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'SVC': SVC(kernel='linear', random_state=42)
}

best_model = None
best_accuracy = 0

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

print(f"Best Model: {best_model.__class__.__name__} with accuracy {best_accuracy:.4f}")

RandomForest Accuracy: 0.7805
GradientBoosting Accuracy: 0.7642
LogisticRegression Accuracy: 0.7886
SVC Accuracy: 0.7236
Best Model: LogisticRegression with accuracy 0.7886


In [18]:
# Function to predict loan eligibility
def predict_loan_eligibility(input_data):
    input_df = pd.DataFrame([input_data], columns=X.columns)
    prediction = best_model.predict(input_df)
    return "Eligible" if prediction[0] == 1 else "Not Eligible"

In [ ]:
# User input for test case
user_input = {}
user_input['Gender'] = int(input("Enter Gender (0 for Female, 1 for Male): "))
user_input['Married'] = int(input("Enter Married (0 for No, 1 for Yes): "))
user_input['Dependents'] = int(input("Enter Dependents (0, 1, 2, 3): "))
user_input['Education'] = int(input("Enter Education (0 for Graduate, 1 for Not Graduate): "))
user_input['Self_Employed'] = int(input("Enter Self Employed (0 for No, 1 for Yes): "))
user_input['ApplicantIncome'] = float(input("Enter Applicant Income: "))
user_input['CoapplicantIncome'] = float(input("Enter Coapplicant Income: "))
user_input['LoanAmount'] = float(input("Enter Loan Amount: "))
user_input['Loan_Amount_Term'] = float(input("Enter Loan Amount Term: "))
user_input['Credit_History'] = int(input("Enter Credit History (0 or 1): "))
user_input['Property_Area'] = int(input("Enter Property Area (0 for Rural, 1 for Semiurban, 2 for Urban): "))

# Derived features
user_input['Total_Income'] = user_input['ApplicantIncome'] + user_input['CoapplicantIncome']
user_input['Total_Income_log'] = np.log(user_input['Total_Income'] + 1)
user_input['LoanAmount_log'] = np.log(user_input['LoanAmount'] + 1)

print(f"Loan Eligibility: {predict_loan_eligibility(user_input)}")